<a href="https://colab.research.google.com/github/sanjaliroy/berkeley-homes-wildfire-agent-simulation/blob/main/wildfire_notes_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homeowners Interview Notes Pre-processing

Course: INFO 290: Fundamentals of Generative AI (Spring 2026)

Creator: Amrita Nambiar

**Variant:** For interviews where participants were **not recorded** — notes were taken by multiple interviewers, so formatting varies across documents.

What this notebook covers:
1. **Notes Ingestion:** Reads `.docx` notes files (varied formatting — bullet points, headers, freeform prose).
2. **Name Extraction:** Derives the interviewee name from document content (e.g. `Interviewee: Heidi`) rather than filename, since notes files may not follow a naming convention.
3. **AI-Driven Extraction:** Uses Gemini 2.5 Flash to transform notes into structured profiles.
4. **Data Export:** Saves results to Google Drive as `agents_from_notes.yaml` and `notes_extraction_debug.jsonl`.

Key differences from the transcript pipeline:
- No interviewer-turn filtering (notes don't have speaker timestamps)
- Name is extracted from document content, not filename
- `notable_quotes` field is dropped — notes are paraphrased, not verbatim
- A pre-pass normalisation step asks Gemini to summarise the notes into a clean paragraph before structured extraction, to handle formatting variance across note-takers

## Install Dependencies

In [1]:
!pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 4.7 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata, drive
from pathlib import Path

import re
import json
import yaml
import google.generativeai as genai
from docx import Document
import pandas as pd

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## Configuration

In [3]:
# Google Drive folder with interview notes (.docx files)
notes_folder_path = "/content/drive/MyDrive/INFO 290: Intro to Gen AI/interview_notes"

# Output folder (same as transcript pipeline — outputs merge into the same directory)
output_folder_path = "/content/drive/MyDrive/INFO 290: Intro to Gen AI/outputs"

# Output filenames (separate from transcript outputs to avoid overwriting)
agents_yaml_filename = "agents_from_notes.yaml"
debug_jsonl_filename = "notes_extraction_debug.jsonl"

# Model
model = "models/gemini-2.5-flash"
max_tokens = 2000

print("Configuration completed")
print(f"Notes folder  : {notes_folder_path}")
print(f"Output folder : {output_folder_path}")
print(f"Model         : {model}")

Configuration completed
Notes folder  : /content/drive/MyDrive/INFO 290: Intro to Gen AI/interview_notes
Output folder : /content/drive/MyDrive/INFO 290: Intro to Gen AI/outputs
Model         : models/gemini-2.5-flash


## Mount Google Drive & Load API Key

In [4]:
# Mount Google Drive
drive.mount("/content/drive")
print("Google Drive mounted")

# Load Gemini API key (same secret as the transcript pipeline)
wildfire_api_key = userdata.get('wildfire')

if wildfire_api_key:
    os.environ["GOOGLE_API_KEY"] = wildfire_api_key
    genai.configure(api_key=wildfire_api_key)
    print("API key loaded.")
else:
    print("API key not found.")

# Verify notes folder exists
notes_dir = Path(notes_folder_path)
if not notes_dir.exists():
    print(f"\nNotes folder not found: {notes_folder_path}")
else:
    docx_files = sorted(notes_dir.glob("*.docx"))
    print(f"\nFound {len(docx_files)} .docx notes file(s):")
    for f in docx_files:
        print(f"   • {f.name}")

Mounted at /content/drive
Google Drive mounted
API key loaded.

Found 9 .docx notes file(s):
   • Candace.docx
   • Greg.docx
   • Heidi.docx
   • Katarina.docx
   • Matt.docx
   • Max.docx
   • Ruth.docx
   • Shah.docx
   • anonymous_1.docx


## Helper Functions

In [5]:
# Extract plain text from .docx
def extract_text_from_docx(docx_path: Path) -> str:
    """Read all paragraphs from a .docx file and return as plain text."""
    doc = Document(str(docx_path))
    paragraphs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
    return "\n\n".join(paragraphs)

In [6]:
# Extract interviewee name from document content
def extract_name_from_notes(text: str, filename_stem: str) -> str:
    """
    Try to find the interviewee name in the document text before falling back
    to the filename stem.

    Looks for common note-taking patterns:
      - 'Interviewee: Heidi'
      - 'Participant: Heidi'
      - 'Name: Heidi'
      - A leading header like '# Heidi' or '## Heidi Interview'
    """
    # Pattern 1: labelled field (case-insensitive)
    label_match = re.search(
        r'^(?:interviewee|participant|name|subject|respondent)\s*[:\-]\s*([A-Z][a-z]+)',
        text, re.IGNORECASE | re.MULTILINE
    )
    if label_match:
        return label_match.group(1).strip()

    # Pattern 2: Markdown heading that looks like a person's name (1-2 words)
    heading_match = re.search(
        r'^#{1,3}\s+([A-Z][a-z]+(\s[A-Z][a-z]+)?)(?:\s|$)',
        text, re.MULTILINE
    )
    if heading_match:
        candidate = heading_match.group(1).strip()
        # Exclude generic headings
        generic = {'notes', 'interview', 'summary', 'background', 'overview'}
        if candidate.lower() not in generic:
            return candidate

    # Pattern 3: First word of the first non-empty line if it looks like a name
    first_line = text.strip().split('\n')[0].strip()
    first_word_match = re.match(r'^([A-Z][a-z]{2,})$', first_line)
    if first_word_match:
        return first_word_match.group(1)

    # Fallback: use the filename stem, cleaned up
    name = filename_stem.split(" - ")[0].split("_")[0].strip()
    return name


print("Name extraction function defined")

Name extraction function defined


In [7]:
# Pre-pass: normalise varied notes into a clean summary
# This handles the formatting variance across different note-takers before
# structured extraction, improving consistency of the final profiles.

normalise_prompt = """\
You are helping pre-process raw interview notes for a wildfire mitigation research project.

The notes below were taken by a researcher during an interview with a Berkeley Hills homeowner.
Note-taking style varies — some notes use bullet points, some use headers, some are freeform prose.

Your task: rewrite the notes as a single, clean, third-person prose summary (4-8 paragraphs)
that preserves ALL factual details, opinions, and specifics mentioned.
Do NOT invent details. Do NOT omit any facts, even minor ones.
Organise loosely into: background/context → property & hardening actions → concerns & barriers →
financial situation → social/community → any other notable points.

RAW NOTES:
{notes}

Respond with the clean prose summary only — no headers, no preamble.
"""


def normalise_notes(raw_notes: str) -> str:
    """Ask Gemini to rewrite varied notes into a consistent prose summary."""
    gemini_model_client = genai.GenerativeModel(model)
    prompt = normalise_prompt.format(notes=raw_notes)
    response = gemini_model_client.generate_content(prompt)
    return response.text.strip()


print("Normalisation function defined")

Normalisation function defined


In [8]:
# Extraction prompt — same schema as transcript pipeline, with two changes:
#   1. Framing changed from "transcript" to "interview notes"
#   2. notable_quotes field removed (notes are paraphrased, not verbatim)

extraction_prompt = """\
You are a research assistant helping build realistic agent personas for a wildfire \
mitigation simulation set in the Berkeley Hills, California.

Below are notes from an interview with a Berkeley Hills homeowner. The notes were taken \
by a researcher (not a verbatim transcript). Your job is to extract a structured profile \
that will be used to initialise a simulation agent.

INTERVIEW NOTES:
{notes}

Extract the following and respond ONLY with valid JSON (no markdown, no preamble, \
no trailing text):

{{
  "agent_id": "<firstname_lowercase — e.g. ross, yen, beth>",
  "display_name": "<First name only — e.g. Ross>",
  "archetype": "<The Resourceful DIYer | The Engaged Community Member | The Skeptical Homeowner | The Compliant Citizen | The Overwhelmed Resident>",
  "age": "<age or age range — e.g. 35>",
  "location": "<Berkeley Hills (Berkeley Woods Firewise area) | specific neighborhood or general area>",
  "persona": "<3-5 sentence paragraph in third person capturing: who they are, \
how long they have lived there, their attitude toward fire risk, their relationship \
with the Ember ordinance and defensible space requirements, their trust in government \
and fire institutions, and any distinctive personality traits revealed in the notes>",
  "risk_zone": "<high | medium | low — infer from location: ridge/hilltop=high, \
mid-hill=medium, flatter/lower=low>",
  "tilden_proximity": "<front_line | mid_hills | moderate_distance | far | unknown>",
  "property_type": "<single_family | rental | residential | unknown>",
  "lot_size": "<large | standard | small | unknown>",
  "roof_type": "<tile | composition_shingle | metal | clay | slate | concrete | wood | asphalt | unknown>",
  "siding": "<stucco | wood | fiber_cement | unknown>",
  "eaves_vents": "<addressed | not_addressed | unknown>",
  "vegetation_zone0": "<fully_compliant | partial | non_compliant | unknown>",
  "vegetation_zone1": "<fully_compliant | partial | non_compliant | unknown>",
  "fence_gates": "<metal | wood | unknown>",
  "institutional_trust": "<high | medium | low — based on how they talk about the \
fire department, county, inspectors, and the Ember ordinance>",
  "compliance_status": "<compliant | in_progress | non_compliant | partial | unknown>",
  "insurance_status": "<retained | non_renewed | dropped | unknown>",
  "years_at_property": <integer or null>,
  "firewise_group": "<name of firewise group or null>",
  "firewise_role": "<active_member | passive_member | leader | unknown | null>",
  "spending_so_far": "<minimal | moderate | high | unknown>",
  "anticipated_costs": "<minimal | moderate | high | unknown>",
  "financial_flexibility": "<high | moderate | low | unknown>",
  "social_network": "<active | limited | inactive | unknown>",
  "action_driver": "<list down what initiatives could potentially help them to take wildfire mitigation actions, it can be \
  grants, resources, discounts, trusted contractors, better guidance, etc, or mark unknown if it wasn't discussed>",
  "key_motivation": "<list down what motivates them to take wildfire mitigation actions, it can be \
  to demonstrate compliance can look good, avoid_penalties, protect their property, avoid losing their insurance, \
  social_awareness, etc or mark unknown if it wasn't discussed>",
  "diy_orientation": "<high | medium | low | unknown>",
  "information_seeking": "<obsessive | moderate | limited | unknown>",
  "emotional_style": "<creative_and_determined | anxious_and_frustrated | resigned_but_compliant | skeptical_and_resistant | unknown>",
  "communication_preferences": "<official_mail | news_media | social | \
direct_experience — pick the single channel type that best matches how they say \
they receive and trust information>",
  "seed_narrative": "<A 3-5 sentence, multi-line paragraph that captures the agent's core story, motivations, and current situation regarding wildfire mitigation.>",
  "core_beliefs": [
    "<belief 1>",
    "<belief 2>",
    "<belief 3>"
  ],
  "memory_seeds": [
    {{
      "description": "<specific first-person memory grounded in the notes — past tense, \
concrete detail, e.g. 'I replaced my wooden fence after the 2023 fire season'>",
      "importance": <integer 1-10>,
      "type": "<observation | reflection | conversation>"
    }},
    {{
      "description": "<another memory>",
      "importance": <integer 1-10>,
      "type": "<observation | reflection | conversation>"
    }},
    {{
      "description": "<another memory>",
      "importance": <integer 1-10>,
      "type": "<observation | reflection | conversation>"
    }},
    {{
      "description": "<another memory>",
      "importance": <integer 1-10>,
      "type": "<observation | reflection | conversation>"
    }},
    {{
      "description": "<another memory>",
      "importance": <integer 1-10>,
      "type": "<observation | reflection | conversation>"
    }}
  ],
  "held_out_responses": {{
    "insurance_non_renewal": {{
      "real_response": "<Concise response if faced with insurance non-renewal>",
      "context": "<Brief context for this response>"
    }},
    "official_regulatory_notice": {{
      "real_response": "<Concise response if faced with an official regulatory notice>",
      "context": "<Brief context for this response>"
    }}
  }},
  "key_concerns": ["<concern 1>", "<concern 2>", "<concern 3>"]
}}

Rules:
- memory_seeds MUST be exactly 5 items, specific to THIS person, not generic
- persona must reflect actual personality from the notes, not a generic homeowner
- Do NOT invent facts not supported by the notes
- If a field is unknown, use null
- notable_quotes is intentionally omitted — notes are paraphrased, not verbatim
- Respond with JSON only — no explanation, no markdown fences
"""


def extract_agent_profile(notes: str, name: str) -> tuple:
    """Send normalised notes to Gemini and return (parsed_dict, raw_text)."""
    # Truncate very long notes to stay within context budget
    words = notes.split()
    if len(words) > 6000:
        notes = ' '.join(words[:6000]) + "\n\n[notes truncated for length]"

    prompt = extraction_prompt.format(notes=notes)

    gemini_model_client = genai.GenerativeModel(model)
    response = gemini_model_client.generate_content(prompt)

    raw_text = response.text.strip()

    # Strip markdown fences if the model adds them despite instructions
    raw_text = re.sub(r'^```(?:json)?\s*', '', raw_text)
    raw_text = re.sub(r'\s*```$', '', raw_text).strip()

    try:
        profile = json.loads(raw_text)
    except json.JSONDecodeError as e:
        print(f"JSON parse error for {name}: {e}")
        profile = {"agent_id": name.lower(), "parse_error": str(e), "raw": raw_text}

    return profile, raw_text


print("Extraction prompt and function defined")

Extraction prompt and function defined


In [9]:
# Convert profile to agents.yaml format
# Identical to the transcript pipeline except notable_quotes is omitted
def profile_to_agent_entry(profile: dict) -> dict:
    """Reshape extracted profile into the agents.yaml schema."""
    if "parse_error" in profile:
        return {"id": profile.get("agent_id", "unknown"), "error": profile["parse_error"]}

    return {
        "id":                      profile.get("agent_id", "unknown"),
        "display_name":            profile.get("display_name", "Unknown"),
        "archetype":               profile.get("archetype"),
        "age":                     profile.get("age"),
        "location":                profile.get("location"),
        "persona":                 profile.get("persona", ""),
        "risk_zone":               profile.get("risk_zone", "unknown"),
        "tilden_proximity":        profile.get("tilden_proximity"),
        "property_type":           profile.get("property_type", "unknown"),
        "lot_size":                profile.get("lot_size"),
        "roof_type":               profile.get("roof_type"),
        "siding":                  profile.get("siding"),
        "eaves_vents":             profile.get("eaves_vents"),
        "vegetation_zone0":        profile.get("vegetation_zone0"),
        "vegetation_zone1":        profile.get("vegetation_zone1"),
        "fence_gates":             profile.get("fence_gates"),
        "institutional_trust":     profile.get("institutional_trust", "medium"),
        "compliance_status":       profile.get("compliance_status", "unknown"),
        "insurance_status":        profile.get("insurance_status"),
        "years_at_property":       profile.get("years_at_property"),
        "firewise_group":          profile.get("firewise_group"),
        "firewise_role":           profile.get("firewise_role"),
        "spending_so_far":         profile.get("spending_so_far"),
        "anticipated_costs":       profile.get("anticipated_costs"),
        "financial_flexibility":   profile.get("financial_flexibility"),
        "social_network":          profile.get("social_network"),
        "action_driver":           profile.get("action_driver"),
        "key_motivation":          profile.get("key_motivation"),
        "diy_orientation":         profile.get("diy_orientation"),
        "information_seeking":     profile.get("information_seeking"),
        "emotional_style":         profile.get("emotional_style"),
        "communication_preferences": profile.get("communication_preferences", "social"),
        "seed_narrative":          profile.get("seed_narrative", ""),
        "core_beliefs":            profile.get("core_beliefs", []),
        "memory_seeds":            profile.get("memory_seeds", []),
        "held_out_responses":      profile.get("held_out_responses", {}),
        "key_concerns":            profile.get("key_concerns", []),
        # notable_quotes intentionally absent — notes are paraphrased
    }


print("Helper functions defined")

Helper functions defined


## Run Extraction (All Notes Files)

In [10]:
notes_dir = Path(notes_folder_path)
output_dir = Path(output_folder_path)
output_dir.mkdir(parents=True, exist_ok=True)

docx_files = sorted(notes_dir.glob("*.docx"))
print(f"Processing {len(docx_files)} notes files...\n")
print("=" * 60)

agents = []
debug_entries = []
errors = []

for i, docx_path in enumerate(docx_files, 1):
    print(f"[{i}/{len(docx_files)}] {docx_path.name}")

    try:
        # Extract raw text from .docx
        raw_text = extract_text_from_docx(docx_path)
        word_count = len(raw_text.split())
        print(f"         {word_count:,} words extracted")

        # Derive interviewee name from content (not filename)
        name = extract_name_from_notes(raw_text, docx_path.stem)
        print(f"         Detected name: {name}")

        # Pre-pass: normalise varied formatting into clean prose
        print(f"         Normalising notes...")
        normalised = normalise_notes(raw_text)
        norm_wc = len(normalised.split())
        print(f"         → {norm_wc:,} words after normalisation")

        # Main extraction pass
        print(f"         Extracting profile...")
        profile, raw_response = extract_agent_profile(normalised, name)

        # Convert to agents.yaml format
        agent_entry = profile_to_agent_entry(profile)
        agents.append(agent_entry)

        # Save debug info
        debug_entries.append({
            "file":               docx_path.name,
            "detected_name":      name,
            "word_count_raw":     word_count,
            "word_count_normalised": norm_wc,
            "normalised_notes":   normalised,
            "raw_llm_response":   raw_response,
            "parsed_profile":     profile
        })

        # Print summary row
        print(f"         agent_id={agent_entry.get('id')} | "
              f"risk={agent_entry.get('risk_zone')} | "
              f"trust={agent_entry.get('institutional_trust')} | "
              f"compliance={agent_entry.get('compliance_status')}")

    except Exception as e:
        print(f"         ERROR: {e}")
        errors.append({"file": docx_path.name, "error": str(e)})

    print()

print("=" * 60)
print(f"Done. {len(agents)} agents extracted, {len(errors)} errors.")

Processing 9 notes files...

[1/9] Candace.docx
         167 words extracted
         Detected name: Candace
         Normalising notes...
         → 283 words after normalisation
         Extracting profile...


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2876.57ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 22877.57ms


         agent_id=candace | risk=high | trust=high | compliance=compliant

[2/9] Greg.docx
         1,059 words extracted
         Detected name: Greg
         Normalising notes...


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 12960.83ms


         → 1,023 words after normalisation
         Extracting profile...
         agent_id=martha | risk=high | trust=medium | compliance=in_progress

[3/9] Heidi.docx
         334 words extracted
         Detected name: Heidi
         Normalising notes...
         → 458 words after normalisation
         Extracting profile...


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 810.63ms


         agent_id=heidi | risk=high | trust=low | compliance=in_progress

[4/9] Katarina.docx
         502 words extracted
         Detected name: Katarina
         Normalising notes...
         → 428 words after normalisation
         Extracting profile...


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 838.82ms


         agent_id=katarina | risk=high | trust=medium | compliance=compliant

[5/9] Matt.docx
         354 words extracted
         Detected name: Matt
         Normalising notes...
         → 421 words after normalisation
         Extracting profile...
         agent_id=matt | risk=high | trust=low | compliance=None

[6/9] Max.docx
         425 words extracted
         Detected name: Max
         Normalising notes...
         → 471 words after normalisation
         Extracting profile...
         agent_id=alex | risk=low | trust=low | compliance=non_compliant

[7/9] Ruth.docx
         462 words extracted
         Detected name: Ruth
         Normalising notes...
         → 481 words after normalisation
         Extracting profile...
         agent_id=ruth | risk=high | trust=high | compliance=compliant

[8/9] Shah.docx
         336 words extracted
         Detected name: Mrs
         Normalising notes...
         → 441 words after normalisation
         Extracting profile...
         

## Save Outputs to Google Drive

In [11]:
# Save agents_from_notes.yaml
agents_yaml_path = output_dir / agents_yaml_filename
with open(agents_yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(
        {"agents": agents},
        f,
        default_flow_style=False,
        allow_unicode=True,
        sort_keys=False
    )

print(f"Saved {len(agents)} agents → {agents_yaml_path}")

# Save notes_extraction_debug.jsonl
debug_jsonl_path = output_dir / debug_jsonl_filename

with open(debug_jsonl_path, "w", encoding="utf-8") as f:
    for entry in debug_entries:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"Saved debug log    → {debug_jsonl_path}")

if errors:
    print(f"\n{len(errors)} file(s) failed:")
    for e in errors:
        print(f"   • {e['file']}: {e['error']}")

Saved 8 agents → /content/drive/MyDrive/INFO 290: Intro to Gen AI/outputs/agents_from_notes.yaml
Saved debug log    → /content/drive/MyDrive/INFO 290: Intro to Gen AI/outputs/notes_extraction_debug.jsonl

1 file(s) failed:
   • anonymous_1.docx: Package not found at '/content/drive/MyDrive/INFO 290: Intro to Gen AI/interview_notes/anonymous_1.docx'


## Summary Table

In [12]:
rows = []
for a in agents:
    rows.append({
        "ID":                      a.get("id", "?"),
        "Agent":                   a.get("display_name", "?"),
        "Archetype":               a.get("archetype", "?"),
        "Age":                     a.get("age", "?"),
        "Location":                a.get("location", "?"),
        "Persona Present":         "Yes" if a.get("persona") else "No",
        "Risk Zone":               a.get("risk_zone", "?"),
        "Tilden Proximity":        a.get("tilden_proximity", "?"),
        "Property Type":           a.get("property_type", "?"),
        "Lot Size":                a.get("lot_size", "?"),
        "Roof Type":               a.get("roof_type", "?"),
        "Siding":                  a.get("siding", "?"),
        "Eaves Vents":             a.get("eaves_vents", "?"),
        "Veg Zone 0":              a.get("vegetation_zone0", "?"),
        "Veg Zone 1":              a.get("vegetation_zone1", "?"),
        "Fence Gates":             a.get("fence_gates", "?"),
        "Trust":                   a.get("institutional_trust", "?"),
        "Compliance":              a.get("compliance_status", "?"),
        "Insurance Status":        a.get("insurance_status", "?"),
        "Years at Property":       a.get("years_at_property", "?"),
        "Firewise Group":          a.get("firewise_group", "?"),
        "Firewise Role":           a.get("firewise_role", "?"),
        "Spending So Far":         a.get("spending_so_far", "?"),
        "Anticipated Costs":       a.get("anticipated_costs", "?"),
        "Financial Flexibility":   a.get("financial_flexibility", "?"),
        "Social Network":          a.get("social_network", "?"),
        "Action Driver":           a.get("action_driver", "?"),
        "Key Motivation":          a.get("key_motivation", "?"),
        "DIY Orientation":         a.get("diy_orientation", "?"),
        "Information Seeking":     a.get("information_seeking", "?"),
        "Emotional Style":         a.get("emotional_style", "?"),
        "Comms Channel":           a.get("communication_preferences", "?"),
        "Narrative Present":       "Yes" if a.get("seed_narrative") else "No",
        "Core Beliefs Count":      len(a.get("core_beliefs", [])),
        "Memory Seeds Count":      len(a.get("memory_seeds", [])),
        "Held Out Responses Present": "Yes" if a.get("held_out_responses", {}) else "No",
        "Key Concerns Count":      len(a.get("key_concerns", [])),
        # notable_quotes column absent — not extracted from notes
    })

df = pd.DataFrame(rows)
print("\nEXTRACTED AGENT SUMMARY")
display(df)

print("\nDISTRIBUTIONS")
for col in df.columns:
    print(f"\n{col}:")
    print(df[col].value_counts().to_string())


EXTRACTED AGENT SUMMARY


,ID,Agent,Archetype,Age,Location,Persona Present,Risk Zone,Tilden Proximity,Property Type,Lot Size,...,Key Motivation,DIY Orientation,Information Seeking,Emotional Style,Comms Channel,Narrative Present,Core Beliefs Count,Memory Seeds Count,Held Out Responses Present,Key Concerns Count
0,candace,Candace,The Resourceful DIYer,None,Grizzly Peak area,Yes,high,front_line,single_family,None,...,"[protect_property, avoid_losing_insurance, avo...",high,obsessive,anxious_and_frustrated,direct_experience,Yes,3,5,Yes,4
1,martha,Martha,The Overwhelmed Resident,None,Berkeley Hills,Yes,high,front_line,single_family,unknown,...,"avoid_penalties, protect their property",unknown,moderate,anxious_and_frustrated,official_mail,Yes,3,5,Yes,4
2,heidi,Heidi,The Engaged Community Member,None,Berkeley Hills,Yes,high,front_line,single_family,None,...,protect their property,high,moderate,anxious_and_frustrated,direct_experience,Yes,4,5,Yes,5
3,katarina,Katarina,The Resourceful DIYer,65+,Berkeley Hills,Yes,high,mid_hills,single_family,None,...,"[protect their property, avoid losing their in...",high,moderate,creative_and_determined,direct_experience,Yes,4,5,Yes,3
4,matt,Matt,The Resourceful DIYer,None,Berkeley Hills,Yes,high,mid_hills,residential,None,...,[protect their property],high,limited,anxious_and_frustrated,direct_experience,Yes,4,5,Yes,3
5,alex,Alex,The Skeptical Homeowner,None,"Lafayette, CA",Yes,low,far,single_family,None,...,[avoid losing their insurance],low,moderate,anxious_and_frustrated,direct_experience,Yes,4,5,Yes,4
6,ruth,Ruth,The Engaged Community Member,None,Berkeley Hills (Firewise program leader),Yes,high,front_line,single_family,None,...,[community resilience against wildfire threats...,medium,moderate,creative_and_determined,direct_experience,Yes,4,5,Yes,4
7,shah,None,The Resourceful DIYer,None,"Alamo, California",Yes,high,None,single_family,None,...,"[protect their property, peace of mind, safety]",high,moderate,creative_and_determined,direct_experience,Yes,4,5,Yes,3



DISTRIBUTIONS

ID:
ID
candace     1
martha      1
heidi       1
katarina    1
matt        1
alex        1
ruth        1
shah        1

Agent:
Agent
Candace     1
Martha      1
Heidi       1
Katarina    1
Matt        1
Alex        1
Ruth        1

Archetype:
Archetype
The Resourceful DIYer           4
The Engaged Community Member    2
The Overwhelmed Resident        1
The Skeptical Homeowner         1

Age:
Age
65+    1

Location:
Location
Berkeley Hills                              4
Grizzly Peak area                           1
Lafayette, CA                               1
Berkeley Hills (Firewise program leader)    1
Alamo, California                           1

Persona Present:
Persona Present
Yes    8

Risk Zone:
Risk Zone
high    7
low     1

Tilden Proximity:
Tilden Proximity
front_line    4
mid_hills     2
far           1

Property Type:
Property Type
single_family    7
residential      1

Lot Size:
Lot Size
unknown    1

Roof Type:
Roof Type
unknown    3

Siding:
Siding
unkno

## Inspect Individual Agent Profile

In [13]:
# Change this to the agent_id you want to inspect
inspect_agent = "heidi"

match = next((a for a in agents if a.get("id") == inspect_agent), None)

if match is None:
    print(f"Agent '{inspect_agent}' not found. Available: {[a.get('id') for a in agents]}")
else:
    print(f"=== {match['display_name'].upper()} ===")
    print(f"\nPERSONA:")
    print(f"  {match['persona']}")

    print(f"\nKEY ATTRIBUTES:")
    for key in [
        'archetype', 'age', 'location', 'risk_zone', 'tilden_proximity',
        'property_type', 'lot_size', 'roof_type', 'siding', 'eaves_vents',
        'vegetation_zone0', 'vegetation_zone1', 'fence_gates',
        'institutional_trust', 'compliance_status', 'insurance_status',
        'years_at_property', 'firewise_group', 'firewise_role',
        'spending_so_far', 'anticipated_costs', 'financial_flexibility',
        'social_network', 'action_driver', 'key_motivation',
        'diy_orientation', 'information_seeking', 'emotional_style',
        'communication_preferences'
    ]:
        print(f"  {key:<28}: {match.get(key)}")

    print(f"\nSEED MEMORIES:")
    for i, mem in enumerate(match.get('memory_seeds', []), 1):
        print(f"  {i}. {mem.get('description', 'N/A')}")

    print(f"\nKEY CONCERNS:")
    for concern in match.get('key_concerns', []):
        print(f"  • {concern}")

=== HEIDI ===

PERSONA:
  Heidi is a long-term Berkeley Hills homeowner and a veteran who shifted from passive observation to proactive home hardening after experiencing nearby wildfires. She is deeply concerned about fire risk and has personally undertaken significant mitigation efforts, referencing Firewise guidelines. While personally committed to defensible space, she expresses low trust in city outreach and the Fire Department due to conflicting advice and a perceived 'double standard' in public land management, often overspending on her efforts. She is also keenly aware of the emotional, aesthetic, and financial barriers that hinder broader community adoption, advocating for clearer guidance and contractor knowledge.

KEY ATTRIBUTES:
  archetype                   : The Engaged Community Member
  age                         : None
  location                    : Berkeley Hills
  risk_zone                   : high
  tilden_proximity            : front_line
  property_type          

## Preview agents_from_notes.yaml Output

In [14]:
# Print the first 2 agents from the YAML to verify formatting
preview = {"agents": agents[:2]}
print(yaml.dump(preview, default_flow_style=False, allow_unicode=True, sort_keys=False))

agents:
- id: candace
  display_name: Candace
  archetype: The Resourceful DIYer
  age: null
  location: Grizzly Peak area
  persona: Candace is a diligent and proactive homeowner residing in the high-risk
    Grizzly Peak area of the Berkeley Hills. Her home, rebuilt in 2015, is a recognized
    'best-in-class' example of wildfire mitigation, fully complying with EMBER requirements
    and updated fire codes. Despite her significant investment and exemplary compliance,
    she faces profound frustration and financial insecurity due to the cancellation
    of her fire insurance and the inadequate coverage of the California FAIR Plan.
    She is determined and actively seeking solutions by navigating the complex insurance
    market herself.
  risk_zone: high
  tilden_proximity: front_line
  property_type: single_family
  lot_size: null
  roof_type: unknown
  siding: unknown
  eaves_vents: addressed
  vegetation_zone0: fully_compliant
  vegetation_zone1: fully_compliant
  fence_gates: u

## Re-run a Single Agent

Use this if one extraction failed or produced a poor result.

In [15]:
# Set to the filename you want to re-run (just the filename, not the full path)
rerun_file = "Heidi.docx"

rerun_path = notes_dir / rerun_file

if not rerun_path.exists():
    print(f"File not found: {rerun_path}")
else:
    raw_text = extract_text_from_docx(rerun_path)
    name = extract_name_from_notes(raw_text, rerun_path.stem)
    print(f"Re-running extraction for: {name}")

    normalised = normalise_notes(raw_text)
    profile, raw_response = extract_agent_profile(normalised, name)
    agent_entry = profile_to_agent_entry(profile)

    # Replace in agents list if already exists, otherwise append
    existing_ids = [a.get("id") for a in agents]
    if agent_entry.get("id") in existing_ids:
        idx = existing_ids.index(agent_entry.get("id"))
        agents[idx] = agent_entry
        print(f"Replaced existing entry for '{agent_entry.get('id')}'")
    else:
        agents.append(agent_entry)
        print(f"Added new entry for '{agent_entry.get('id')}'")

    print(f"\nResult:")
    print(json.dumps(agent_entry, indent=2))

    # Re-save the YAML
    with open(agents_yaml_path, "w", encoding="utf-8") as f:
        yaml.dump({"agents": agents}, f, default_flow_style=False,
                  allow_unicode=True, sort_keys=False)
    print(f"\nUpdated {agents_yaml_path}")

Re-running extraction for: Heidi


TooManyRequests: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 52.526965884s.